# InvoiceGuard Round 2 - Training Reproducibility Notebook

**Hackathon:** OpenEnv Hackathon 2026 (Round 2)  
**Author:** piyush-mk  
**Model:** Qwen/Qwen3-4B-Instruct-2507 (bf16 LoRA)

This notebook is the full reproducibility path used for InvoiceGuard Round 2 training. It is designed for judges and contributors to run step-by-step in a fresh GPU runtime and verify both framework compliance and real project training behavior.

## Notebook structure

1. **Runtime setup and secret checks**
   - install required Python packages,
   - validate that `HF_TOKEN` and `HF_USERNAME` are present.
2. **Framework coverage section (required checklist item)**
   - minimal Hugging Face TRL training run,
   - Unsloth availability check.
3. **Primary project training path (used for reported results)**
   - submit-focused SFT run,
   - artifact and metric verification,
   - warm-start GRPO from the SFT adapter.
4. **Interpretation and troubleshooting guidance**
   - expected files,
   - expected score trends,
   - common failure fixes.

## Recommended runtime

- Colab or Kaggle with GPU enabled
- T4/P100 minimum, L4/A10G preferred
- `HF_TOKEN` with write access to your account

## Baseline expectation

This notebook includes the same training strategy used in the repository story. The strongest SFT result from this path reaches around `0.70+` grader score with substantial improvement over the local untrained baseline.

## Quick start

1. Start a fresh GPU runtime.
2. Set secrets (`HF_TOKEN`, `HF_USERNAME`) in the notebook environment.
3. Run cells in order from top to bottom.
4. Open your Hub repo to verify adapter + metrics artifacts after each stage.

In [ ]:
%pip -q install -U uv huggingface_hub

In [ ]:
import os

# Auto-detect HF_TOKEN from Colab or Kaggle secrets
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or os.environ.get("HF_TOKEN", "")
except Exception:
    pass
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN") or os.environ.get("HF_TOKEN", "")
except Exception:
    pass

os.environ.setdefault("HF_USERNAME", "piyush-mk")
os.environ.setdefault("INVOICEGUARD_CODE_REPO", "piyush-mk/invoiceguard-code")
os.environ.setdefault("BASE_MODEL", "Qwen/Qwen3-4B-Instruct-2507")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN before running."
print(f"Model: {os.environ['BASE_MODEL']}")
print(f"Code repo: {os.environ['INVOICEGUARD_CODE_REPO']}")

In [ ]:
import torch
print(f"torch {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM: {free/1e9:.1f} / {total/1e9:.1f} GB")

## 0.5 TRL Compliance Mini-Run (Required Checklist)

This section satisfies the framework requirement with a minimal TRL training example that is small enough to run quickly in Colab.

### What this section proves

- The runtime can install and import TRL correctly.
- A real `SFTTrainer` run executes end-to-end.
- The notebook includes explicit framework usage beyond static code snippets.

### Scope

This is a compliance sanity run, not the primary project benchmark. It uses tiny synthetic data so it finishes quickly and avoids consuming most of the session budget before the main InvoiceGuard training path.

In [ ]:
# Minimal TRL example (tiny, fast, compliance-oriented)
# Runs a very small SFTTrainer job on synthetic data to prove TRL setup works.

%pip -q install -U trl transformers datasets accelerate peft

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer

mini_model = "Qwen/Qwen2.5-0.5B-Instruct"

samples = [
    {"text": "You are InvoiceGuard. Return JSON only.\nTask: quantity mismatch with received < billed.\nAnswer: {\"action_type\":\"submit_final_resolution\",\"final_decision\":\"place_on_hold\",\"exception_type\":\"partial_receipt\",\"evidence_references\":[\"compare_quantity\"],\"explanation\":\"Billed quantity exceeds received quantity.\"}"},
    {"text": "You are InvoiceGuard. Return JSON only.\nTask: duplicate invoice found in prior paid record.\nAnswer: {\"action_type\":\"submit_final_resolution\",\"final_decision\":\"reject_invoice\",\"exception_type\":\"duplicate_invoice\",\"evidence_references\":[\"check_for_duplicate_invoice\"],\"explanation\":\"Invoice was already processed.\"}"},
    {"text": "You are InvoiceGuard. Return JSON only.\nTask: unit price exceeds PO tolerance.\nAnswer: {\"action_type\":\"submit_final_resolution\",\"final_decision\":\"escalate_for_supervisor_review\",\"exception_type\":\"price_mismatch\",\"evidence_references\":[\"compare_price\",\"inspect_policy_rules\"],\"explanation\":\"Price variance exceeds tolerance.\"}"},
    {"text": "You are InvoiceGuard. Return JSON only.\nTask: all docs match and policy checks pass.\nAnswer: {\"action_type\":\"submit_final_resolution\",\"final_decision\":\"approve_for_payment\",\"exception_type\":\"clean_match\",\"evidence_references\":[\"compare_totals\"],\"explanation\":\"No discrepancy detected.\"}"},
]

train_ds = Dataset.from_list(samples)

tokenizer = AutoTokenizer.from_pretrained(mini_model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(mini_model)

args = TrainingArguments(
    output_dir="./trl_compliance_artifacts",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    max_steps=4,
    learning_rate=2e-5,
    logging_steps=1,
    save_steps=4,
    report_to=[],
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field="text",
    args=args,
)

trainer.train()
print("TRL compliance mini-run complete.")

### Unsloth coverage check

This check confirms whether the current runtime can import Unsloth.

- If import succeeds, the notebook records that Unsloth is available in this environment.
- If import fails, the notebook continues normally because the required TRL path is already executed.

This keeps the notebook robust across Colab/Kaggle runtimes while still documenting framework coverage clearly.

In [ ]:
# Unsloth availability check
import subprocess, sys

print("Installing Unsloth...")
res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth"], capture_output=True, text=True)
if res.returncode == 0:
    try:
        import unsloth  # noqa: F401
        print("Unsloth is available in this runtime.")
    except Exception as e:
        print(f"Unsloth install succeeded but import failed: {e}")
else:
    print("Unsloth is not available in this runtime. TRL compliance run above remains valid.")

## 1. Submit-Focused SFT (Primary Training Stage)

This is the main training stage used in the project results. The idea is to train the missing completion behavior directly: the base model already investigates reasonably, but often fails to finalize with `submit_final_resolution`.

### Why submit-focused SFT

In full-trace SFT, investigation actions dominate token count and can bias the model toward endless investigation loops. Submit-focused SFT shifts learning signal toward the final decision behavior while still preserving context from prior steps.

### Core flags and rationale

- `--no-4bit`
  - uses bf16-style path for better stability and quality.
- `--submit-only`
  - trains only resolution actions.
- `--min-investigation-steps 7`
  - keeps richer context before submission so the model learns to conclude after evidence gathering.
- `--max-new-tokens 384`
  - avoids truncating structured JSON outputs.

### Expected runtime behavior

- early epochs may remain close to baseline,
- then score and success can jump sharply once termination behavior is learned,
- best checkpoint may occur before the final epoch, so artifact verification matters.

In [ ]:
!HF_TOKEN="$HF_TOKEN" \
HF_USERNAME="$HF_USERNAME" \
INVOICEGUARD_CODE_REPO="$INVOICEGUARD_CODE_REPO" \
BASE_MODEL="$BASE_MODEL" \
HUB_MODEL_ID="invoiceguard-qwen3-4b-sft-notebook" \
TRACKIO_PROJECT="invoiceguard-round2" \
TRACKIO_RUN_NAME="sft-notebook" \
PYTORCH_CUDA_ALLOC_CONF="$PYTORCH_CUDA_ALLOC_CONF" \
TOKENIZERS_PARALLELISM="$TOKENIZERS_PARALLELISM" \
uv run https://huggingface.co/$INVOICEGUARD_CODE_REPO/resolve/main/training/train_sft.py \
  --no-4bit \
  --submit-only \
  --min-investigation-steps 7 \
  --num-epochs 12 \
  --eval-holdout-canonical 2 \
  --eval-holdout-hard 2 \
  --lr 5e-5 \
  --max-new-tokens 384 \
  --max-prompt-tokens 2048

## 2. Verify SFT Results and Artifacts

After SFT completes, verify both artifact presence and metric trend.

### What to verify

1. **Adapter files exist** in the target Hub repo.
2. **Metrics JSONL is present** and parseable.
3. **Eval lines show improvement** over local untrained behavior.

### What good output looks like

- Adapter files such as `adapter_model.safetensors` and `adapter_config.json` are visible.
- `sft_artifacts/sft_metrics.jsonl` contains multiple epoch rows.
- `eval/avg_grader_score` and `eval/success_rate` improve meaningfully versus early rows.

If artifacts are missing, check token permissions and repository naming first.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
import json

REPO = f"{os.environ['HF_USERNAME']}/invoiceguard-qwen3-4b-sft-notebook"
api = HfApi(token=os.environ["HF_TOKEN"])

files = api.list_repo_files(repo_id=REPO, repo_type="model")
print(f"Repo: {REPO}")
print(f"Files: {len(files)}")
for f in sorted(files):
    print(f"  {f}")

# Show metrics
try:
    metrics_path = hf_hub_download(REPO, "sft_artifacts/sft_metrics.jsonl", token=os.environ["HF_TOKEN"])
    print("\n--- Eval Results ---")
    with open(metrics_path) as f:
        for line in f:
            row = json.loads(line)
            if "eval/avg_grader_score" in row:
                print(f"Epoch {row['step']:2d}: score={row['eval/avg_grader_score']:.4f}  "
                      f"success={row['eval/success_rate']:.0%}  "
                      f"steps={row['eval/avg_steps']:.1f}")
except Exception as e:
    print(f"Could not load metrics: {e}")

## 3. GRPO Run (Warm-Started from SFT)

This stage runs GRPO from the SFT adapter rather than from the raw base model.

### Why warm-start

SFT establishes reliable formatting and submission behavior first. GRPO then optimizes rollout quality using environment rewards and grader-driven signals.

### Key settings in this notebook

- `--resume-adapter $HF_USERNAME/invoiceguard-qwen3-4b-sft-notebook`
- `--group-size 8`
- `--sample-temperature 1.3`
- short holdout eval slices for quick iteration

### Reading GRPO outcomes

- Focus on holdout score trend and stability, not just final iteration.
- Best checkpoint can appear before the last iteration.
- If instability appears, keep SFT checkpoint as fallback and select best GRPO checkpoint by eval score.

In [ ]:
# Run GRPO from the SFT checkpoint
!HF_TOKEN="$HF_TOKEN" \
HF_USERNAME="$HF_USERNAME" \
INVOICEGUARD_CODE_REPO="$INVOICEGUARD_CODE_REPO" \
BASE_MODEL="$BASE_MODEL" \
HUB_MODEL_ID="invoiceguard-qwen3-4b-grpo-notebook" \
TRACKIO_PROJECT="invoiceguard-round2" \
TRACKIO_RUN_NAME="grpo-notebook" \
PYTORCH_CUDA_ALLOC_CONF="$PYTORCH_CUDA_ALLOC_CONF" \
TOKENIZERS_PARALLELISM="$TOKENIZERS_PARALLELISM" \
uv run https://huggingface.co/$INVOICEGUARD_CODE_REPO/resolve/main/training/train_grpo.py \
  --no-4bit \
  --no-format-warmup \
  --resume-adapter $HF_USERNAME/invoiceguard-qwen3-4b-sft-notebook \
  --num-iterations 3 \
  --group-size 8 \
  --sample-temperature 1.3 \
  --eval-holdout-canonical 2 \
  --eval-holdout-hard 2 \
  --max-new-tokens 384 \
  --max-prompt-tokens 2048

## Expected Output, Interpretation, and Troubleshooting

A successful run uploads the following to your Hub repository:

| Stage | Artifact type | Files |
|------|---------------|-------|
| SFT | LoRA adapter | `adapter_model.safetensors`, `adapter_config.json` |
| SFT | Metrics | `sft_artifacts/sft_metrics.jsonl` |
| SFT | Summary | `sft_artifacts/sft_summary.json` |
| GRPO | Metrics | `training_artifacts/metrics.jsonl` (path may vary by script version) |
| GRPO | Summaries/rollouts | run-specific JSON outputs in training artifacts |

### Metric interpretation guide

- `eval/avg_grader_score`: primary quality indicator on holdout eval.
- `eval/success_rate`: percent of tasks fully resolved.
- `eval/avg_steps`: whether behavior converges toward efficient completion.

For SFT, a common healthy pattern is movement from low baseline behavior toward `0.6+` score range. For GRPO, improvements may be non-monotonic; checkpoint selection should use best holdout score, not only final step.

### Common issues and quick fixes

- **HF auth failures**: verify `HF_TOKEN` has write scope and repo namespace is correct.
- **OOM / CUDA instability**: reduce group size or use a higher-memory GPU runtime.
- **No metrics uploaded**: verify script finished and artifact upload step executed.
- **Weak GRPO learning**: ensure warm-start adapter is loaded and sampling temperature/group settings are not too conservative.

This notebook is intentionally verbose so each stage can be audited independently by anyone reviewing reproducibility.